# Generaci?n y Clasificaci?n de Secuencias Gen?ticas Sint?ticas

Este notebook explica paso a paso c?mo se genera un dataset sint?tico de secuencias gen?ticas, se entrena un clasificador de riesgo y se utiliza un modelo generativo LSTM para crear nuevas secuencias, todo con Python y librer?as de IA.

## 1. Importaci?n de Librer?as

Se importan las librer?as necesarias para la generaci?n de datos, modelado y clasificaci?n.

In [ ]:
import numpy as np
import random
from faker import Faker
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

fake = Faker()

## 2. Generaci?n de Datos Sint?ticos

Se crean funciones para generar secuencias gen?ticas aleatorias y asignar un nivel de riesgo. Luego, se construye un dataset sint?tico.

In [ ]:
def generar_secuencia_genetica(longitud=8):
    return [random.randint(1, 20) for _ in range(longitud)]

def riesgo_aleatorio():
    # Ahora las clases est?n balanceadas
    return random.choice([0, 1, 2])

# Dataset sint?tico
X = np.array([generar_secuencia_genetica() for _ in range(800)])
y = np.array([riesgo_aleatorio() for _ in range(800)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Entrenamiento del Clasificador RandomForest

Se entrena un clasificador RandomForest para predecir el nivel de riesgo de una secuencia gen?tica.

In [ ]:
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

print(f"RandomForest accuracy test: {clf.score(X_test, y_test):.2f}")

## 4. Preparaci?n y Entrenamiento del Modelo LSTM

Se prepara el dataset para el modelo generativo LSTM, que aprender? a predecir la siguiente base gen?tica en una secuencia.

In [ ]:
vocab_size = 21
max_len = 8
embed_dim = 12

# Preparar secuencias para modelo generativo
sequences = []
next_bases = []
for seq in X:
    for i in range(1, len(seq)):
        sequences.append(seq[:i])
        next_bases.append(seq[i])

sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
next_bases = to_categorical(next_bases, num_classes=vocab_size)

model = Sequential([
    Embedding(vocab_size, embed_dim, input_length=max_len),
    LSTM(64),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')

model.fit(sequences, next_bases, epochs=5, batch_size=32, verbose=0)

## 5. Diccionario de Traducci?n de Bases Gen?ticas

Se define un diccionario para traducir los n?meros de las bases a nombres cient?ficos o ficticios.

In [ ]:
base_map = {
    1:"Adenina",2:"Citosina",3:"Guanina",4:"Timina",5:"Uracilo",
    6:"Elemento X1",7:"Elemento X2",8:"Elemento X3",9:"Elemento X4",10:"Elemento X5",
    11:"Prote?na A",12:"Prote?na B",13:"Enzima C",14:"Enzima D",
    15:"Mutaci?n Alta",16:"Mutaci?n Baja",17:"Secuencia Cr?ptica",
    18:"Secuencia Activa",19:"Elemento Regulador",20:"Elemento Silenciador"
}

## 6. Funciones de Generaci?n y Traducci?n de Secuencias

Se implementan funciones para generar nuevas secuencias con el modelo LSTM y traducirlas a texto legible.

In [ ]:
def generar_secuencia(model, seed_seq, longitud=max_len):
    result = seed_seq.copy()
    for _ in range(longitud - len(seed_seq)):
        padded = pad_sequences([result], maxlen=max_len, padding='pre')
        pred = model.predict(padded, verbose=0)[0]
        next_base = np.random.choice(range(vocab_size), p=pred)
        if next_base == 0:  # padding
            break
        result.append(next_base)
    return result

def secuencia_a_texto(seq):
    return ", ".join(base_map.get(b, "?") for b in seq)

## 7. Clase BioFakerIA

Clase que genera especies ficticias con nombre, h?bitat, descripci?n, genoma sint?tico y nivel de riesgo.

In [ ]:
class BioFakerIA:
    def __init__(self):
        self.fake = Faker()

    def species_name(self):
        return f"{self.fake.last_name()} {self.fake.word().capitalize()}"

    def habitat(self):
        return self.fake.city()

    def description(self):
        descs = [
            "Especie end?mica con adaptaciones ?nicas.",
            "Conocida por su r?pido crecimiento.",
            "Posee mecanismos defensivos avanzados.",
            "Importante en el ecosistema local.",
            "Presenta una dieta variada y flexible."
        ]
        return random.choice(descs)

    def generate(self):
        nombre = self.species_name()
        habitat = self.habitat()
        descripcion = self.description()

        seed = generar_secuencia_genetica(3)
        secuencia = generar_secuencia(model, seed)
        texto_gen = secuencia_a_texto(secuencia)

        # --- Arreglo: Prepara la secuencia para el clasificador ---
        # Quita ceros (padding) y ajusta longitud a 8
        secuencia_sin_ceros = [b for b in secuencia if b != 0]
        if len(secuencia_sin_ceros) < 8:
            # Rellena con bases aleatorias v?lidas
            secuencia_sin_ceros += [random.randint(1, 20) for _ in range(8 - len(secuencia_sin_ceros))]
        elif len(secuencia_sin_ceros) > 8:
            secuencia_sin_ceros = secuencia_sin_ceros[:8]
        # Predice el riesgo usando la secuencia ajustada
        riesgo = clf.predict([secuencia_sin_ceros])[0]
        niveles = {0:"Bajo",1:"Medio",2:"Alto"}

        bio = f"?? Especie: {nombre}\n" \
              f"??? H?bitat: {habitat}\n" \
              f"?? Descripci?n: {descripcion}\n" \
              f"?? Genoma sint?tico: {texto_gen}\n" \
              f"?? Nivel de riesgo para humanos: {niveles[riesgo]}"
        return bio

## 8. Ejemplo de Generaci?n de Especie Sint?tica

Se crea una instancia de la clase y se genera una especie ficticia.

In [ ]:
biofaker = BioFakerIA()
print(biofaker.generate())

## 9. Interfaz Web con Streamlit

Puedes ejecutar la siguiente app para generar especies ficticias desde una interfaz web:

```bash
streamlit run biofaker_app.py
```